In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%run DataPreparation.ipynb

In [41]:
data = dataset.iloc[:, 1:-1].values
target = dataset.iloc[:, -1].values

train_data, test_data, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

# MODEL SELECTION

## Support Vector Machine

COARSE SEARCH

In [3]:
svr = SVR()

parameters_SVR = {'kernel': ['rbf', 'sigmoid'], 'C': [0.1, 1, 10], 'gamma': [1e-3, 1e-2]}

search = GridSearchCV(svr, param_grid=parameters_SVR)

search.fit(train_data, train_target)

GridSearchCV(estimator=SVR(),
             param_grid={'C': [0.1, 1, 10], 'gamma': [0.001, 0.01],
                         'kernel': ['rbf', 'sigmoid']})

FINE SEARCH

In [4]:
svr.set_params(kernel='rbf')

parameters_fine_SVR = {'C':[9, 10, 11], 'gamma': [1e-3, 2e-3, 5e-4]}

search_fine = GridSearchCV(svr, param_grid=parameters_fine_SVR)

search_fine.fit(train_data, train_target)

GridSearchCV(estimator=SVR(),
             param_grid={'C': [9, 10, 11], 'gamma': [0.001, 0.002, 0.0005]})

In [14]:
svr.set_params(C=11, gamma=0.0005)

scores = cross_val_score(svr, train_data, train_target, cv=5)
print(scores)
print(np.mean(scores))
print(np.std(scores))

[-0.04620033 -0.11409101 -0.01557072 -0.03276817 -0.054281  ]
-0.05258224448161482
0.03343469304856249


## XGBoost

COARSE SEARCH

In [15]:
xgb = XGBRegressor(n_jobs=-1)

parameters_XGB = {'n_estimators': [100,500,1000], 'max_depth': [3, 7, 10], 'learning_rate': [1e-2, 1e-3], 'gamma': [0, 0.1, 0.3, 1]}

search = GridSearchCV(xgb, param_grid=parameters_XGB, verbose=1, n_jobs=-1)

search.fit(train_data, train_target)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


GridSearchCV(estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,
                                    lear...
                                    max_cat_threshold=None,
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=-1, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'gamma': [0, 0.1, 0.3, 1],
                         'learning_rate': [0.01, 0.001],
                         'max_depth': [3, 7, 10],
                         'n_estimators': [100, 500, 1000]},
             verbose=1)

FINE SEARCH

In [16]:
xgb.set_params(gamma=search.best_params_['gamma'],max_depth=search.best_params_['max_depth'])

fine_parameters_XGB = {'n_estimators':[1000, 1500, 2000], 'learning_rate':[0.01, 0.05, 0.07]}

fine_search = GridSearchCV(xgb, param_grid=fine_parameters_XGB, verbose=1, n_jobs=-1)

fine_search.fit(train_data, train_target)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


GridSearchCV(estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=0, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,
                                    learning_rate=None, max_bin=None,
                                    max_cat_threshold=None,
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=3, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=-1, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.07],
                         'n_estimators': [1000, 1500, 2000]},
             verbose=1)

In [17]:
xgb.set_params(n_estimators=fine_search.best_params_['n_estimators'], learning_rate=fine_search.best_params_['learning_rate'])

scores = cross_val_score(xgb, train_data, train_target, cv=5, scoring='r2')
print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.83507907 0.77013379 0.87695813 0.88485068 0.91042703]
0.8554897427558898
0.04907809261123959


## Random Forest

Coarse Search

In [29]:
rand_f = RandomForestRegressor(n_jobs=-1)

parameters_RandF = {'n_estimators': [100,200,500], 'max_depth': [None, 10, 20, 30], 'max_features': ['sqrt', 'log2', 0.5], 'min_samples_split': [2, 5, 10, 20], 'bootstrap':[True, False]}

search = GridSearchCV(rand_f, param_grid=parameters_RandF, n_jobs=-1)

search.fit(train_data, train_target)

GridSearchCV(estimator=RandomForestRegressor(n_jobs=-1), n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'max_depth': [None, 10, 20, 30],
                         'max_features': ['sqrt', 'log2', 0.5],
                         'min_samples_split': [2, 5, 10, 20],
                         'n_estimators': [100, 200, 500]})

Fine search

In [31]:
rand_f.set_params(bootstrap=search.best_params_['bootstrap'], max_features=search.best_params_['max_features'], min_samples_split=search.best_params_['min_samples_split'])

fine_parameteres_RandF = {'max_depth':[30, 40, 50], 'n_estimators':[50, 75, 100]}

fine_search = GridSearchCV(rand_f, param_grid=fine_parameteres_RandF, n_jobs=-1)

fine_search.fit(train_data, train_target)

GridSearchCV(estimator=RandomForestRegressor(bootstrap=False,
                                             max_features='sqrt', n_jobs=-1),
             n_jobs=-1,
             param_grid={'max_depth': [30, 40, 50],
                         'n_estimators': [50, 75, 100]})

In [32]:
rand_f.set_params(n_estimators=fine_search.best_params_['n_estimators'], max_depth=fine_search.best_params_['max_depth'])

scores = cross_val_score(rand_f, train_data, train_target, cv=5, scoring='r2')
print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.8548249  0.78600408 0.8250936  0.88171949 0.90358644]
0.8502457029589332
0.04151245055515076


## AdaBoost

Coarse search

In [37]:
adaBR = AdaBoostRegressor()

parameters_AdaB = {'n_estimators': [50,100,200,400], 'learning_rate': [0.01, 0.05, 0.1, 0.001], 'loss': ['linear', 'square', 'exponential'], 
                   'estimator':[
                       DecisionTreeRegressor(max_depth=1),
                       DecisionTreeRegressor(max_depth=2),
                       DecisionTreeRegressor(max_depth=3)
                   ]}

search = GridSearchCV(adaBR, param_grid=parameters_AdaB, n_jobs=-1)

search.fit(train_data, train_target)

c:\Users\vmm\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\ma\core.py:2846: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


GridSearchCV(estimator=AdaBoostRegressor(), n_jobs=-1,
             param_grid={'estimator': [DecisionTreeRegressor(max_depth=1),
                                       DecisionTreeRegressor(max_depth=2),
                                       DecisionTreeRegressor(max_depth=3)],
                         'learning_rate': [0.01, 0.05, 0.1, 0.001],
                         'loss': ['linear', 'square', 'exponential'],
                         'n_estimators': [50, 100, 200, 400]})

In [39]:
adaBR.set_params(n_estimators=search.best_params_['n_estimators'], estimator=search.best_params_['estimator'], learning_rate=search.best_params_['learning_rate'], loss=search.best_params_['loss'])

scores = cross_val_score(adaBR, train_data, train_target, cv=5, scoring='r2')
print(scores)
print(np.mean(scores))
print(np.std(scores))

[0.81145568 0.67384019 0.77466973 0.80938285 0.80889272]
0.7756482353838434
0.05270906665105665


## Model: XGBoost & Random Forest

In [ ]:
xgb.fit(train_data, train_target)

predx = xgb.predict(test_data)

print(f"XGBoost R² score: {r2_score(test_target, predx)}")
print(f"XGBoost MAE score: {mean_absolute_error(test_target, predx)}")
print(f"XGBoost MSE score: {mean_squared_error(test_target, predx)}")

XGBoost R² score: 0.9130996465682983
XGBoost MAE score: 16044.831670055652
XGBoost MSE score: 666554214.8871615


In [ ]:
rand_f.fit(train_data, train_target)

predr = rand_f.predict(test_data)

print(f"Random Forest R² score: {r2_score(test_target, predr)}")
print(f"Random Forest MAE score: {mean_absolute_error(test_target, predr)}")
print(f"Random Forest MSE score: {mean_squared_error(test_target, predr)}")

Random Forest R² score: 0.8903519611923371
Random Forest MAE score: 17023.681780821917
Random Forest MSE score: 841036225.6330878
